In [4]:
!pip install -U langgraph langchain langchain_openai tavily-python yfinance langchain_community

import os
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
import yfinance as yf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: yfinance
    Found existing installation: yfinance 0.2.66
    Uninstalling yfinance-0.2.66:
      Succes

In [32]:
# --- 1. STATE DEFINITION ---
from google.colab import userdata
TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
class TradingState(TypedDict):
    ticker: str
    technical_analysis: str
    news_sentiment: str
    final_signal: str
    user_approval: str # HITL feedback
    status: str

In [39]:
# --- 2. CUSTOM TOOLS ---
@tool
def get_stock_price(ticker: str):
    """Fetches the latest closing price and basic stats for a ticker."""
    stock = yf.Ticker(ticker)
    hist = stock.history(period="30d")
    return f"Latest Price: {hist['Close'].iloc[-1]:.2f}. 5-Day Trend: {hist['Close'].tolist()}"

search_tool = TavilySearchResults()

In [34]:



# --- 3. AGENTS ---
model = ChatOpenAI(model="gpt-4o-mini",api_key= OPENAI_API_KEY, temperature=0)

def technical_analyst(state: TradingState):
    price_data = get_stock_price.invoke(state["ticker"])
    return {"technical_analysis": price_data, "status": "analyzing_news"}

def sentiment_analyst(state: TradingState):
    search_query = f"latest news sentiment for {state['ticker']} stock"
    news = search_tool.invoke(search_query)
    return {"news_sentiment": str(news), "status": "awaiting_approval"}

def generate_final_signal(state: TradingState):
    prompt = f"Based on Tech: {state['technical_analysis']} and News: {state['news_sentiment']}, provide a final Buy/Sell/Hold recommendation. Provide a detailed explanation, broken down into multiple paragraphs."
    res = model.invoke(prompt)
    return {"final_signal": res.content, "status": "complete"}

In [35]:
# --- 4. GRAPH CONSTRUCTION ---
workflow = StateGraph(TradingState)

workflow.add_node("tech_analyst", technical_analyst)
workflow.add_node("news_analyst", sentiment_analyst)
workflow.add_node("human_gate", lambda state: state) # HITL Pause Point
workflow.add_node("generate_signal", generate_final_signal) # New node

workflow.set_entry_point("tech_analyst")
workflow.add_edge("tech_analyst", "news_analyst")
workflow.add_edge("news_analyst", "human_gate")

# Function to route based on user approval
def route_decision(state: TradingState):
    if state.get("user_approval") == "approved":
        return "generate_signal"
    else:
        return END

workflow.add_conditional_edges("human_gate", route_decision)
workflow.add_edge("generate_signal", END) # End after generating the signal

# Compile with Interruption
app = workflow.compile(checkpointer=MemorySaver(), interrupt_before=["human_gate"])

In [36]:
# --- 5. CORE FUNCTION ---
def execute_workflow(ticker: str, thread_id="trade_1"):
    config = {"configurable": {"thread_id": thread_id}}
    app.invoke({"ticker": ticker, "user_approval": "", "status": "starting"}, config)

    state = app.get_state(config).values
    print(f"\n[PAUSED] Analysis for {ticker} is ready.")
    print(f"Technical: {state['technical_analysis']}")
    print("Please approve to see the final signal.")

    feedback = input("Type 'approved' to get the signal, or 'rejected' to stop: ")
    resume_workflow(feedback, thread_id)

In [37]:
def resume_workflow(feedback: str, thread_id="trade_1"):
    config = {"configurable": {"thread_id": thread_id}}

    # 2. Update the state with the human's feedback
    app.update_state(config, {"user_approval": feedback}, as_node="human_gate")

    print(f"--- Resuming workflow with feedback: '{feedback}' ---")

    # Explicitly invoke the app from the human_gate node to finish the workflow.
    # We do NOT use stream_mode here to avoid verbose output from internal LangGraph logging.
    app.invoke(None, config=config)

    # Retrieve the final state after the workflow has resumed and completed
    final_state_snapshot = app.get_state(config).values

    if final_state_snapshot.get("final_signal"):
        print("\n==================================")
        print("✅ FINAL TRADE SIGNAL:")
        print("==================================")
        # The print function will automatically handle newlines within the string
        print(final_state_snapshot["final_signal"])
        print("==================================")
    elif final_state_snapshot.get("status") == "complete":
        print("\nWorkflow Finished.")
    else:
        print("\nWorkflow did not complete as expected or no final signal was generated.")
        print(f"Current state: {final_state_snapshot}")


In [38]:
execute_workflow('MSFT')


[PAUSED] Analysis for MSFT is ready.
Technical: Latest Price: 413.84. 5-Day Trend: [424.82000732421875, 429.25, 424.4599914550781, 407.7799987792969, 413.8399963378906]
Please approve to see the final signal.
Type 'approved' to get the signal, or 'rejected' to stop: approved
--- Resuming workflow with feedback: 'approved' ---

✅ FINAL TRADE SIGNAL:
Based on the latest data regarding Microsoft Corporation (MSFT), the stock is currently priced at $413.84, reflecting a slight recovery from a recent dip. The 5-day trend shows fluctuations, with the stock peaking at $429.25 and dipping to $407.78, indicating some volatility in the short term. The recent news surrounding Microsoft presents a mixed picture, with strong earnings reports highlighting significant revenue growth, particularly in AI and cloud services, but also concerns regarding competition and declining revenues in certain segments.

### Earnings Performance
Microsoft's recent fiscal Q3 results were impressive, with a reported 

In [18]:
execute_workflow('Gold')



[PAUSED] Analysis for Gold is ready.
Technical: Latest Price: 43.69. 5-Day Trend: [47.209999084472656, 45.720001220703125, 45.16999816894531, 45.189998626708984, 43.69499969482422]
Please approve to see the final signal.
Type 'approved' to get the signal, or 'rejected' to stop: approved


--- Resuming workflow with feedback: 'approved' ---

Workflow did not complete as expected or no final signal was generated.
Current state: {'ticker': 'Gold', 'technical_analysis': 'Latest Price: 43.69. 5-Day Trend: [47.209999084472656, 45.720001220703125, 45.16999816894531, 45.189998626708984, 43.69499969482422]', 'news_sentiment': '[{\'title\': "Gold News - Today\'s Breaking Gold News | Investing.com", \'url\': \'https://www.investing.com/commodities/gold-news\', \'content\': \'| Name | Last | Chg. % | Vol. |  |\\n ---  --- \\n| INTC | 98.93 | +4.71% | 73.01M |  |\\n| NVDA | 199.18 | -0.20% | 51.59M |  |\\n| AAPL | 282.06 | +3.95% | 36.15M |  |\\n| TSLA | 390.56 | +2.34% | 21.73M |  |\\n| AMZN | 268.85 | +1.43% | 19.70M |  |\\n| MU | 525.20 | +1.55% | 19.51M |  |\\n| SNDK | 1,081.39 | -1.38% | 10.86M |  |\\n\\n| Name | Last | Chg. % | Vol. |  |\\n ---  --- \\n| PSKY | 11.15 | +8.84% | 4.01M |  |\\n| CBOE | 322.51 | +7.47% | 563.91K |  |\\n| MSTR | 176.22 | +6.51% | 7.30M |  |\\n| EL 

### Test Case 1: Apple (AAPL) - Approved

In [40]:
# Simulate user input 'approved'
execute_workflow('AAPL', thread_id='trade_2') # User will input 'approved'


[PAUSED] Analysis for AAPL is ready.
Technical: Latest Price: 283.86. 5-Day Trend: [247.99000549316406, 251.49000549316406, 251.63999938964844, 252.6199951171875, 252.88999938964844, 248.8000030517578, 246.6300048828125, 253.7899932861328, 255.6300048828125, 255.9199981689453, 258.8599853515625, 253.5, 258.8999938964844, 260.489990234375, 260.4800109863281, 259.20001220703125, 258.8299865722656, 266.42999267578125, 263.3999938964844, 270.2300109863281, 273.04998779296875, 266.1700134277344, 273.1700134277344, 273.42999267578125, 271.05999755859375, 267.6099853515625, 270.7099914550781, 270.1700134277344, 271.3500061035156, 283.8599853515625]
Please approve to see the final signal.
Type 'approved' to get the signal, or 'rejected' to stop: approved
--- Resuming workflow with feedback: 'approved' ---

✅ FINAL TRADE SIGNAL:
### Recommendation: Buy

#### Current Market Position
As of the latest data, Apple Inc. (AAPL) is trading at $283.86, which reflects a significant increase from its re

### Test Case 2: Tesla (TSLA) - Rejected

In [ ]:
# Simulate user input 'rejected'
execute_workflow('TSLA', thread_id='trade_3') # User will input 'rejected'

### Test Case 3: Google (GOOG) - Approved

In [ ]:
# Simulate user input 'approved'
execute_workflow('GOOG', thread_id='trade_4') # User will input 'approved'

### Test Case 4: Pfizer (PFE) - Approved

In [ ]:
# Simulate user input 'approved'
execute_workflow('PFE', thread_id='trade_5') # User will input 'approved'

### Test Case 5: Bitcoin (BTC-USD) - Approved

In [ ]:
# Simulate user input 'approved'
execute_workflow('BTC-USD', thread_id='trade_6') # User will input 'approved'